# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Imports
import duckdb
from getpass import getpass

# Authenticate with Hugging Face
token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

Enter your Hugging Face READ token: ··········


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

---

### Feature Vector

The feature vector is built from the March 2026 analysis window established during ML-04.

Only features available at prediction time are included. Availability flags are retained so missing integrations are not confused with genuine zero values. Numeric metrics are filled where appropriate using `COALESCE`, while `gsc_avg_position` is intentionally left unchanged because missing values carry information about search data availability.

In [8]:
feature_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    COALESCE(client_has_gsc, FALSE) AS client_has_gsc,
    COALESCE(client_has_ga4, FALSE) AS client_has_ga4,
    COALESCE(gsc_data_available, FALSE) AS gsc_data_available,
    COALESCE(ga4_data_available, FALSE) AS ga4_data_available,

    COALESCE(gsc_impressions, 0) AS gsc_impressions,
    COALESCE(gsc_clicks, 0) AS gsc_clicks,
    gsc_avg_position,

    COALESCE(ga4_pageviews, 0) AS ga4_pageviews,
    COALESCE(ga4_engaged_sessions, 0) AS ga4_engaged_sessions

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)

WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
print("Feature vector shape:", feature_df.shape)

feature_df.head()

Feature vector shape: (9841378, 12)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,False,20,0,3.350000,0,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,False,1,0,0.000000,0,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,False,125,1,4.928000,0,0
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,False,7,0,4.000000,0,0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,False,11,0,2.272727,0,0


In [10]:
feature_df.isna().sum()

,0
report_date,0
client_hash_id,0
content_hash_id,0
client_has_gsc,0
client_has_ga4,0
gsc_data_available,0
ga4_data_available,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,6230317


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

---

### Feature Notes

| Feature | Meaning | Missing Handling | Available Before Prediction? |
|---------|---------|------------------|------------------------------|
| `report_date` | Reporting date for the observation. | None. | Yes |
| `client_hash_id` | Anonymous client identifier. | None. | Yes |
| `content_hash_id` | Anonymous content/page identifier. | None. | Yes |
| `client_has_gsc` | Indicates whether the client has Google Search Console connected. | Missing values are filled with `False` using `COALESCE`. | Yes |
| `client_has_ga4` | Indicates whether the client has Google Analytics 4 connected. | Missing values are filled with `False` using `COALESCE`. | Yes |
| `gsc_data_available` | Indicates Search Console data availability for the observation. | Missing values are filled with `False` using `COALESCE`. | Yes |
| `ga4_data_available` | Indicates GA4 data availability for the observation. | Missing values are filled with `False` using `COALESCE`. | Yes |
| `gsc_impressions` | Search impressions recorded by Google Search Console. | Missing values are filled with `0` using `COALESCE`. | Yes |
| `gsc_clicks` | Search clicks recorded by Google Search Console. | Missing values are filled with `0` using `COALESCE`. | Yes |
| `gsc_avg_position` | Average Google Search position for the page. | Missing values are intentionally retained because they indicate unavailable ranking information. | Yes |
| `ga4_pageviews` | Pageviews recorded by Google Analytics 4. | Missing values are filled with `0` using `COALESCE`. | Yes |
| `ga4_engaged_sessions` | Engaged sessions recorded by Google Analytics 4. | Missing values are filled with `0` using `COALESCE`. | Yes |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

---

### Leakage Hunt

The feature vector was reviewed for common sources of data leakage before any modeling.

The following checks were performed:

- No label-derived columns were included.
- All features come from the March 2026 analysis window only.
- No future observations or future aggregations were used.
- Client and content identifiers are retained for traceability only and are excluded from model training.
- Product availability flags describe information available at prediction time and are not derived from future outcomes.

In [11]:
print("Columns used in the feature vector:\n")

for col in feature_df.columns:
    print("-", col)

Columns used in the feature vector:

- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions


In [13]:
leakage_patterns = [
    "label",
    "target",
    "future",
    "next",
    "outcome",
    "action",
    "refresh_flag",
    "recommended_action",
]

suspect_columns = [
    col for col in feature_df.columns
    if any(pattern in col.lower() for pattern in leakage_patterns)
]

print("Leakage Audit\n")

if suspect_columns:
    print("Potential leakage columns found:")
    for col in suspect_columns:
        print("-", col)
else:
    print("✓ No obvious label-derived or future-information columns detected.")

print("\nFeature window: March 2026 only")
print("✓ No future partitions included.")

Leakage Audit

✓ No obvious label-derived or future-information columns detected.

Feature window: March 2026 only
✓ No future partitions included.


### Observation

No obvious sources of target leakage were identified.

The feature vector contains identifiers, availability flags, and performance metrics observed within the March 2026 analysis window. No label-derived columns, future observations, or post-outcome variables were included.

Based on these checks, the feature set is suitable for baseline modeling without introducing obvious data leakage.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

---

### What I Excluded and Why

| Excluded field | Why |
|---------------|-----|
| `report_date` | Defines the observation date but is not used as a predictive feature. |
| `client_hash_id` | Unique identifier. Excluding it reduces memorization of client-specific patterns. |
| `content_hash_id` | Unique identifier. Excluding it encourages learning generalizable behavior instead of page-specific IDs. |

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.